In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
# !pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

zsh:1: command not found: pip
zsh:1: command not found: pip
2.21.0


In [3]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

--2026-11-10 13:12:08--  https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 104.26.2.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 358233 (350K) [text/tab-separated-values]
Saving to: ‘train-data.tsv’

train-data.tsv      100%[===================>] 349,84K  --.-KB/s    in 0,09s   

2026-11-10 13:12:09 (3,69 MB/s) - ‘train-data.tsv’ saved [358233/358233]

--2026-11-10 13:12:09--  https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 172.67.70.149, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK


In [9]:
# We create the first datasets and then also give them a label and message header

train_dataset = pd.read_csv(train_file_path,sep="\t",names=["label","message"])

test_dataset = pd.read_csv(test_file_path, sep="\t",names=["label","message"])

# Then we nned to one hot encode ham and spame with ham = 0 and spam = 1

train_dataset["label"] = train_dataset["label"].map({"ham" : 0 ,"spam":1})

test_dataset["label"] = test_dataset["label"].map({"ham" : 0 ,"spam":1})

print(train_dataset.head())
print(train_dataset["label"].value_counts())
print(train_dataset.isna().sum())

   label                                            message
0      0  ahhhh...just woken up!had a bad dream about u ...
1      0                           you can never do nothing
2      0  now u sound like manky scouse boy steve,like! ...
3      0  mum say we wan to go then go... then she can s...
4      0  never y lei... i v lazy... got wat? dat day ü ...
label
0    3619
1     560
Name: count, dtype: int64
label      0
message    0
dtype: int64


In [20]:
# We split the train_messages and what not 

train_msgs = train_dataset["message"]
train_label = train_dataset["label"]

test_msgs = test_dataset["message"]
test_label = test_dataset["label"]


# We also need to change their datatypes and make the msges into lists of strings

train_msgs_list = np.array(train_msgs.tolist())
test_msgs_list = np.array(test_msgs.tolist())


In [22]:
# we se tthe params 

VOCAB_SIZE = 2000
SEQ_LENGTH = 50

# then we vectorize the text

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=SEQ_LENGTH
)

vectorizer.adapt(train_msgs)

In [ ]:

# Some printing 

print(vectorizer.get_vocabulary()[:20])
print(vectorizer(["Win a free prize now!"]))


['', '[UNK]', np.str_('to'), np.str_('i'), np.str_('you'), np.str_('a'), np.str_('the'), np.str_('u'), np.str_('and'), np.str_('in'), np.str_('is'), np.str_('me'), np.str_('my'), np.str_('for'), np.str_('your'), np.str_('of'), np.str_('it'), np.str_('call'), np.str_('have'), np.str_('on')]
tf.Tensor(
[[185   5  49 163  22   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0]], shape=(1, 50), dtype=int64)


In [23]:
# We Create the model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,),dtype=tf.string),
    vectorizer,
    tf.keras.layers.Embedding(input_dim = VOCAB_SIZE,output_dim=16),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16,activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# We compile the model 

model.compile(loss="binary_crossentropy",
              optimizer = "adam",
              metrics = ["accuracy"])

# then the summary of the model 
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_3            │ (None, 50)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 50, 16)         │        32,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,289 (126.13 KB)

 Trainable params: 32,289 (126.13 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
# The the fitting of the model 
train_msgs_tf = tf.constant(train_msgs_list)
test_msgs_tf = tf.constant(test_msgs_list)

history = model.fit(
    train_msgs_tf,
    np.array(train_label, dtype="float32"),
    epochs=30,
    validation_data=(test_msgs_tf, np.array(test_label, dtype="float32")),
    verbose=2
)

Epoch 1/30
131/131 - 0s - 2ms/step - accuracy: 0.9876 - loss: 0.0447 - val_accuracy: 0.9806 - val_loss: 0.0638
Epoch 2/30
131/131 - 0s - 1ms/step - accuracy: 0.9873 - loss: 0.0410 - val_accuracy: 0.9806 - val_loss: 0.0561
Epoch 3/30
131/131 - 0s - 1ms/step - accuracy: 0.9892 - loss: 0.0365 - val_accuracy: 0.9835 - val_loss: 0.0547
Epoch 4/30
131/131 - 0s - 1ms/step - accuracy: 0.9914 - loss: 0.0334 - val_accuracy: 0.9820 - val_loss: 0.0516
Epoch 5/30
131/131 - 0s - 1ms/step - accuracy: 0.9916 - loss: 0.0313 - val_accuracy: 0.9820 - val_loss: 0.0540
Epoch 6/30
131/131 - 0s - 2ms/step - accuracy: 0.9911 - loss: 0.0291 - val_accuracy: 0.9849 - val_loss: 0.0513
Epoch 7/30
131/131 - 0s - 1ms/step - accuracy: 0.9943 - loss: 0.0273 - val_accuracy: 0.9828 - val_loss: 0.0514
Epoch 8/30
131/131 - 0s - 1ms/step - accuracy: 0.9935 - loss: 0.0246 - val_accuracy: 0.9849 - val_loss: 0.0482
Epoch 9/30
131/131 - 0s - 1ms/step - accuracy: 0.9935 - loss: 0.0233 - val_accuracy: 0.9849 - val_loss: 0.0488
E

In [32]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):

   # 1. Wrap the string in a tf.constant (a batch of one message)
    msg = tf.constant([pred_text])

    # 2. Get the model's prediction: a 2D array like [[0.93]]
    prob = float(model.predict(msg, verbose=0)[0][0])

    # 3. Pick the label with a 0.5 cutoff
    label = "spam" if prob > 0.5 else "ham"

    return [prob, label]

  

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

[0.000117397379653994, 'ham']


In [33]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()


You passed the challenge. Great job!
